In [1]:
# ===== Sentence Window Retrieval (Colab one cell, no API key) =====
!pip -q install -U sentence-transformers

import re
from sentence_transformers import SentenceTransformer
import numpy as np

embed = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# 1) Sample document (can be single file chunked into sentences)
# -----------------------------
doc_text = """
Add/Drop is available in the first two teaching weeks. Students should check the academic calendar for exact dates.
The official deadline is typically Week 2 Friday. Late add/drop requests may require approval by the school office.
A prerequisite is a module you must complete before enrolling in another module. Some modules have co-requisites.
Tuition fees depend on programme and citizenship status. Always verify fees on the official website.
If you are working part-time, allocate fixed study blocks every week. Keep a buffer day before deadlines.
"""

# -----------------------------
# 2) Sentence splitter
# -----------------------------
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', doc_text.strip()) if s.strip()]
sent_embs = embed.encode(sentences, normalize_embeddings=True)

# -----------------------------
# 3) Sentence Window Retriever
# -----------------------------
def sentence_window_retrieve(query: str, window: int = 2, top_k_sentences: int = 1):
    q_emb = embed.encode([query], normalize_embeddings=True)[0]

    # similarity scores between query and each sentence
    scores = sent_embs @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k_sentences]

    results = []
    for idx in top_idx:
        start = max(0, idx - window)
        end = min(len(sentences), idx + window + 1)
        context_window = " ".join(sentences[start:end])

        results.append({
            "best_sentence_index": int(idx),
            "best_sentence": sentences[idx],
            "score": float(scores[idx]),
            "window_text": context_window
        })
    return results

# -----------------------------
# 4) Demo
# -----------------------------
query = "What is the add/drop deadline?"
out = sentence_window_retrieve(query, window=2, top_k_sentences=1)

print("QUERY:", query)
print("\nBest matched sentence:")
print(out[0]["best_sentence"])
print("Score:", out[0]["score"])

print("\n--- Sentence Window (±2 sentences) ---")
print(out[0]["window_text"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

QUERY: What is the add/drop deadline?

Best matched sentence:
The official deadline is typically Week 2 Friday.
Score: 0.6415666937828064

--- Sentence Window (±2 sentences) ---
Add/Drop is available in the first two teaching weeks. Students should check the academic calendar for exact dates. The official deadline is typically Week 2 Friday. Late add/drop requests may require approval by the school office. A prerequisite is a module you must complete before enrolling in another module.
